# Swactor — Getting Started

This notebook walks through the basics of the swactor actor runtime from Python.

In [ ]:
from swactor import Runtime

## Single-threaded: spawn, send, tick

The simplest way to use swactor is with `tick()` — manually stepping the runtime one tick at a time.

In [ ]:
def echo(ctx, msg):
    ctx.send(msg["reply_to"], f"hello, {msg['name']}!")

rt = Runtime()
addr = rt.spawn(echo)
inbox = rt.inbox()

rt.send(addr, {"name": "world", "reply_to": inbox.addr})
rt.tick()

print(inbox.try_recv())

## Multi-threaded: background runtime

For real workloads you can run the runtime on background threads with `RuntimeConfig`.

In [ ]:
import time
from swactor import Runtime, RuntimeConfig

def echo(ctx, msg):
    ctx.send(msg["reply_to"], f"hello, {msg['name']}!")

rt = Runtime(RuntimeConfig(num_threads=2))
addr = rt.spawn(echo)
inbox = rt.inbox()
handle = rt.run()

for name in ["alice", "bob", "charlie"]:
    handle.send(addr, {"name": name, "reply_to": inbox.addr})
    time.sleep(0.05)  # give the runtime a moment
    print(inbox.try_recv())

handle.shutdown()
handle.join()